In [2]:
import pandas as pd 
import numpy as np 


In [3]:
import pandas as pd

df = pd.read_csv("spam.csv", encoding="latin-1")
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [4]:
df=df.drop(['Unnamed: 2','Unnamed: 3','Unnamed: 4'], axis=1)

In [5]:
from sklearn.preprocessing import LabelEncoder

In [6]:
x=df.drop('v1', axis=1)
y=df['v1']

In [7]:
label=LabelEncoder()
y=label.fit_transform(y)

In [8]:
y

array([0, 0, 1, ..., 0, 0, 0], shape=(5572,))

In [9]:
import re
from bs4 import BeautifulSoup
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


KeyboardInterrupt: 

In [ ]:
def clean_text(text):
    text=text.lower()
    ext = BeautifulSoup(text, "html.parser").get_text()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    # Remove special characters and punctuation
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
x['v2']=x['v2'].apply(clean_text)

In [ ]:
tokenizer=Tokenizer()
x['v2']=tokenizer.fit_on_texts(x['v2'])

In [ ]:
sequence=tokenizer.texts_to_sequences(df['v2'])

In [ ]:
import pickle
with open('tokenizer.pkl', 'wb')as file:
    pickle.dump(tokenizer, file)

In [ ]:
print(sequence)

[[46, 443, 4309, 794, 710, 679, 64, 9, 1245, 90, 119, 356, 1246, 154, 2862, 1247, 67, 58, 4310, 137], [48, 312, 1398, 444, 6, 1820], [50, 460, 9, 21, 4, 749, 899, 1, 179, 1821, 1130, 624, 1822, 2196, 261, 2197, 71, 1821, 1, 1823, 1, 313, 460, 606, 1034, 79, 542, 740, 382], [6, 229, 142, 24, 357, 2866, 6, 160, 143, 60, 142], [950, 2, 97, 72, 461, 1, 900, 72, 1824, 202, 111, 462], [844, 120, 67, 1585, 100, 184, 22, 7, 39, 343, 86, 56, 109, 383, 3, 42, 12, 15, 84, 1825, 48, 369, 1034, 4311, 1, 70, 1086, 1, 2867], [203, 11, 598, 8, 25, 56, 1, 359, 34, 10, 108, 680, 10, 56, 4312, 4313], [73, 216, 13, 1131, 1399, 1826, 2198, 2199, 2200, 112, 100, 573, 73, 13, 951, 12, 51, 1586, 952, 463, 1, 1035, 13, 217, 951], [681, 73, 4, 795, 413, 218, 3, 17, 100, 414, 1, 2868, 140, 953, 1, 115, 16, 2869, 115, 401, 2870, 504, 711, 547, 64], [129, 13, 92, 1037, 796, 27, 124, 6, 85, 1132, 1, 487, 1, 5, 303, 548, 845, 34, 329, 12, 50, 16, 5, 92, 487, 1248, 50, 18, 2871], [219, 32, 80, 209, 7, 2, 69, 1, 264, 

In [ ]:
max_lenght=500
x = pad_sequences(
    sequence,
    maxlen=max_lenght,
    padding='pre'
)

In [ ]:
x

array([[   0,    0,    0, ...,   58, 4310,  137],
       [   0,    0,    0, ...,  444,    6, 1820],
       [   0,    0,    0, ...,  542,  740,  382],
       ...,
       [   0,    0,    0, ...,  102,  231, 9412],
       [   0,    0,    0, ...,  198,   12,   50],
       [   0,    0,    0, ...,    1,   41,  258]],
      shape=(5572, 500), dtype=int32)

In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train, y_test=train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
import tensorflow as tf 
from tensorflow.keras.layers import Dense ,SimpleRNN, Embedding
from tensorflow.keras.models import Sequential

In [ ]:
max_feature=10000
model=Sequential()
model.add(Embedding(max_feature, 128, input_length=max_lenght ))
model.add(SimpleRNN(128, activation='relu', return_sequences=True))
model.add(SimpleRNN(64, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

d:\PROJECTS\DEEP LEARNING PROJECT\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
earlystopping=EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
history=model.fit(
    x_train, y_train, epochs=50,
    validation_data=(x_test, y_test),
    callbacks=[earlystopping]
)

Epoch 1/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 31s 220ms/step - accuracy: 1.0000 - loss: 1.3407e-07 - val_accuracy: 0.9848 - val_loss: 0.2568
Epoch 2/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 34s 240ms/step - accuracy: 1.0000 - loss: 1.1405e-07 - val_accuracy: 0.9848 - val_loss: 0.2423
Epoch 3/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 30s 217ms/step - accuracy: 1.0000 - loss: 1.2942e-07 - val_accuracy: 0.9848 - val_loss: 0.2453
Epoch 4/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 30s 213ms/step - accuracy: 1.0000 - loss: 9.8847e-08 - val_accuracy: 0.9848 - val_loss: 0.2525
Epoch 5/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 33s 236ms/step - accuracy: 1.0000 - loss: 1.0761e-07 - val_accuracy: 0.9848 - val_loss: 0.2442
Epoch 6/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 33s 235ms/step - accuracy: 1.0000 - loss: 9.2025e-08 - val_accuracy: 0.9848 - val_loss: 0.2569
Epoch 7/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 32s 231ms/step - accuracy: 1.0000 - loss: 7.8838e-08 - val_accuracy: 0.9848 - val_loss: 0.2518
Epoch 8/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 32s 225ms/step 

In [ ]:
print(x_train.dtype)
print(y_train.dtype)

int32
str


In [ ]:
model.save('sms_spam_model.h5')